# Chapter 04 · Mathematics: Gradient history and momentum

[Chapter guide](README.md) · [Problem-to-mathematics guide](../MATHEMATICS.md) · [Course index](../README.md)

**Status: optional reference for later; not yet reviewed.** The programming lessons are already covered. Continue the programming-led mentorship without needing to complete this companion first. Use small numerical explanations when helpful, and revisit formal notation gradually.

Read one section at a time: words → a small example → symbols → code → your explanation. Symbols are shorthand for ideas you already used. You do not need to memorize all the formulas before continuing.

**Prerequisites:** Chapter 01 math: gradients and SGD updates; Chapter 03 math: checkpoint selection.

The runnable examples use only Python and the installed PyTorch. Start in a fresh kernel and run Setup, then proceed in order. Each example keeps its calculations small and supplies its own data. Code cells are saved unexecuted so you can make your own predictions first.

## Contents

1. [1. Parameters and gradients are different vectors](#sgd)
2. [2. Add remembered gradients to the current direction](#momentum)
3. [3. Mathematical state for resuming training](#state)

## Setup

In [ ]:
import torch

<a id="sgd"></a>
## 1. Parameters and gradients are different vectors


A **vector** is an ordered collection of values. Here $\theta=(w,b)$ groups our parameters, while $g=(g_w,g_b)$ groups their gradients. They contain different kinds of information.

$$\theta_{t+1}=\theta_t-\eta g_t$$

$t$ counts updates. This shorthand applies the familiar scalar update to every registered parameter. A weight gradient of `-20` and bias gradient of `-10`, with learning rate `0.05`, give increases of `1` and `0.5` respectively.

**Connection:** programming Lessons 01–02. **Before running:** does a bigger absolute gradient produce a bigger plain-SGD step when the learning rate is the same?


**My prediction or explanation:** _write here_

In [ ]:
# Vector notation is just the same calculation for each parameter value.
parameters = torch.tensor([1., 0.])
gradients = torch.tensor([-20., -10.])
parameter_changes = -0.05 * gradients
print("Parameter changes [weight, bias]:", parameter_changes.tolist())
print("New parameters:", (parameters + parameter_changes).tolist())

<a id="momentum"></a>
## 2. Add remembered gradients to the current direction


Let $v_t$ mean the stored update-direction buffer after update $t$, and let $\mu$ (mu) be the momentum coefficient. With no weight decay, no dampening, and no Nesterov option, our PyTorch SGD exercise uses:

$$v_t=\mu v_{t-1}+g_t,\qquad \theta_t=\theta_{t-1}-\eta v_t$$

The first buffer is initialized to the first gradient. Writing a conceptual previous buffer of zero gives the same first-step arithmetic here. The buffer is separate from `.grad`: clearing the current gradients does not clear the remembered direction.

At the second weight update, current gradient is `-11.5`, previous buffer is `-20`, and momentum is `0.9`:

$$v_2=0.9(-20)+(-11.5)=-29.5$$

The old weight is `2`, so its new value is $2-0.05(-29.5)=3.475$.

At the next step, the weight gradient can be positive while the remembered buffer remains negative. The weight can therefore keep increasing despite the current gradient suggesting a decrease. This explains overshooting and oscillation in our experiments.

**Connection:** programming Lessons 03–05 and [PyTorch SGD](https://docs.pytorch.org/docs/stable/generated/torch.optim.SGD.html). A different momentum setting is not guaranteed to be better at every update budget.

**Before running:** calculate the second buffer, then check the sign of the third gradient and buffer.


**My prediction or explanation:** _write here_

In [ ]:
# Inspect three real PyTorch SGD steps and the buffer used for each update.
x_values = torch.tensor([0., 1., 2., 3.])
y_values = torch.tensor([2., 5., 8., 11.])
weight = torch.tensor(1., requires_grad=True)
bias = torch.tensor(0., requires_grad=True)
optimizer = torch.optim.SGD([weight, bias], lr=0.05, momentum=0.9)
for step in range(1, 4):
    optimizer.zero_grad(set_to_none=True)
    loss = ((weight * x_values + bias - y_values) ** 2).mean()
    loss.backward()
    fresh_gradient = weight.grad.item()
    optimizer.step()
    direction = optimizer.state[weight]["momentum_buffer"].item()
    print(f"Step {step} | current gradient={fresh_gradient:.4f} | "
          f"buffer={direction:.4f} | weight={weight.item():.4f}")

<a id="state"></a>
## 3. Mathematical state for resuming training


For predictions, the formula needs the model's parameter values. For the next momentum update, it also needs the stored buffer and optimizer settings.

Two models at the same $\theta$ can have the same predictions and current gradients but different next updates if their buffers differ:

$$v_{\mathrm{restored}}=0.9(-4)+(-3.5)=-7.1$$

$$v_{\mathrm{fresh}}=-3.5$$

This is why our training checkpoint stored model state, optimizer state, and progress together. That was sufficient for the deterministic fixed-batch example; more elaborate runs can have additional state such as changing data order.

**Connection:** programming Lesson 06 for selection, and Chapter 05's model-checkpoint notebook for resuming. **Before running:** from weight `1.2`, which next weight comes from each of the two directions?


**My prediction or explanation:** _write here_

In [ ]:
# Hold parameter value, current gradient, and learning rate fixed.
weight_before, current_gradient, learning_rate = 1.2, -3.5, 0.05
restored_direction = 0.9 * (-4.0) + current_gradient
fresh_direction = current_gradient
print("Weight with restored buffer:", weight_before - learning_rate * restored_direction)
print("Weight with fresh buffer:", weight_before - learning_rate * fresh_direction)

**My observation:** _write here_

Distinguish parameter values, fresh gradients, momentum buffers, and hyperparameters such as learning rate. A checkpoint selected for lowest validation loss is a model-selection choice; optimizer-state restoration is a training-continuation choice.